# Axral Vocal AI — Kaggle One-cell pipeline**入力は `HF_DATA_URL` だけ**。Hub revision固定→UTAU関連ファイルだけを容量制限付きで取得→ZIP/UAR/TARを安全展開→文字コードに配慮した音源監査→pytest→承認済みラベルmanifestがある場合の学習・評価→成果物ZIPまで1セルで実行します。> デフォルト上限はダウンロード4096 MiB・展開8192 MiBです。学習対象は現時点の実験用メタデータ・ピッチ差分モデルです。UTAU WAVだけから歌唱表現の正解ラベルは生成できないため、承認済みラベル付きJSONLが無ければ学習を安全に停止します。音声生成モデルの学習ではありません。**Hosted-processing permission gate:** labelled training manifests must explicitly mark `hosted_processing`, `feature_extraction`, and `ml_training` as `approved` for every record. Local permission alone is insufficient for Kaggle. Model distribution and audio redistribution remain independent decisions and are not approved by a successful training run.### One-time Kaggle Secret setup1. Open the notebook's **Secrets** panel and add a secret named `AXRAL_HOSTED_PROCESSING_REVIEWED`.2. Set its value to exactly `yes` and enable access for this notebook.3. Do this only after checking the exact release and current terms for **every voicebank selected by the dataset URL**, and confirming that hosted processing on Kaggle is permitted.The cell checks this secret before it removes old temporary data or downloads anything. A missing or different value stops the run safely. This secret is a human acknowledgement; it does not verify a license automatically. For a private Hugging Face dataset, add a separate `HF_TOKEN` secret; public datasets do not need it.The only editable code setting remains `HF_DATA_URL`. The permission acknowledgement and optional access token are stored as Kaggle secrets, not hard-coded in the notebook.

In [ ]:
# ============================================================
# AXRAL VOCAL AI — ONE CELL KAGGLE PIPELINE
# 設定は HF_DATA_URL だけ。取得→文字コード対策→監査→pytest→条件付き学習・評価→ZIP
# ============================================================
import os, re, sys, json, subprocess, traceback, zipfile, shutil, hashlib
from pathlib import Path

# ★ここだけ変更：Hugging Face の Dataset URL（例 https://huggingface.co/datasets/OWNER/REPO）
HF_DATA_URL = "https://huggingface.co/datasets/SEKOIA2900/VO-US"

# One-time Kaggle Secret: AXRAL_HOSTED_PROCESSING_REVIEWED=yes.
# This acknowledges that every selected voicebank permits processing on Kaggle.
hosted_processing_review = os.environ.get("AXRAL_HOSTED_PROCESSING_REVIEWED", "")
if not hosted_processing_review:
    try:
        from kaggle_secrets import UserSecretsClient
        hosted_processing_review = UserSecretsClient().get_secret(
            "AXRAL_HOSTED_PROCESSING_REVIEWED"
        )
    except Exception:
        hosted_processing_review = ""
if hosted_processing_review.strip().casefold() != "yes":
    raise RuntimeError(
        "Hosted processing is blocked. Review each selected voicebank's terms, "
        "then set Kaggle Secret AXRAL_HOSTED_PROCESSING_REVIEWED=yes only when "
        "Kaggle processing is explicitly permitted."
    )
PROJECT = Path("/kaggle/working/Axral_vocal_AI")
DATA_ROOT = Path("/kaggle/temp/hf_voicebanks")
OUT = Path("/kaggle/working/axral_run")
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)  # prevent stale files from previous dataset/revision
if OUT.exists(): shutil.rmtree(OUT)  # prevent stale reports from previous runs
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "pipeline.log"

def sanitize_dataset_paths(value):
    # Detailed per-bank paths stay in /kaggle/temp and are never copied to saved artifacts.
    return re.sub(re.escape(str(DATA_ROOT)) + r"[^\r\n]*", "<dataset-path>", str(value))

def log(msg):
    safe_msg = sanitize_dataset_paths(msg)
    print(safe_msg)
    with LOG.open("a", encoding="utf-8") as f: f.write(safe_msg + "\n")

def safe_command_diagnostic(output):
    """Keep only stable status/error codes from a JSON CLI response."""
    for line in reversed(output.splitlines()):
        try:
            payload = json.loads(line)
        except (json.JSONDecodeError, TypeError):
            continue
        if not isinstance(payload, dict):
            continue
        diagnostic = {}
        for key in ("status", "error_code"):
            value = payload.get(key)
            if isinstance(value, str) and re.fullmatch(r"[a-z0-9_]{1,80}", value):
                diagnostic[key] = value
        if diagnostic:
            return diagnostic
    return {}


def run(cmd, check=True, output_tail=1200, redact_output=False, env=None):
    log("$ " + " ".join(map(str, cmd)))
    child_env = dict(os.environ if env is None else env)
    if env is None:
        # Never pass a globally configured Hub credential to unrelated commands.
        child_env.pop("HF_TOKEN", None)
    p = subprocess.run(cmd, cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, errors="replace", env=child_env)
    if redact_output:
        diagnostic = safe_command_diagnostic(p.stdout)
        diagnostic_text = json.dumps(diagnostic, ensure_ascii=False, sort_keys=True)
        log(
            f"command exit={p.returncode}; safe diagnostic={diagnostic_text}; "
            "detailed dataset output suppressed"
        )
    else:
        safe_output = sanitize_dataset_paths(p.stdout)
        with LOG.open("a", encoding="utf-8") as f: f.write(safe_output)
        if output_tail and safe_output:
            print(safe_output[-output_tail:])
        else:
            log(f"command exit={p.returncode}; full output saved to pipeline.log")
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode


def is_trusted_axral_origin(remote_urls):
    """Accept only the canonical Axral Vocal AI GitHub origin."""
    trusted_origins = {
        "https://github.com/nebula-INE/Axral_vocal_AI",
        "https://github.com/nebula-INE/Axral_vocal_AI.git",
        "git@github.com:nebula-INE/Axral_vocal_AI",
        "git@github.com:nebula-INE/Axral_vocal_AI.git",
        "ssh://git@github.com/nebula-INE/Axral_vocal_AI",
        "ssh://git@github.com/nebula-INE/Axral_vocal_AI.git",
    }
    return len(remote_urls) == 1 and remote_urls[0] in trusted_origins


LOG.write_text("", encoding="utf-8")
try:
    assert HF_DATA_URL.startswith("https://huggingface.co/"), "HF_DATA_URL must be a Hugging Face URL"
    required_pipeline_files = (
        "scripts/kaggle_hf_download.py",
        "scripts/voicebank_audit_summary.py",
        "training/train_pitch_model.py",
        "src/axral_vocal_ai/voicebank_audit_cli.py",
    )
    if not PROJECT.exists():
        log("Project checkout missing; cloning Axral_vocal_AI default branch.")
        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/nebula-INE/Axral_vocal_AI.git", str(PROJECT)],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, errors="replace"
        )
        log(clone.stdout)
        if clone.returncode:
            raise RuntimeError(f"Git clone failed ({clone.returncode})")
    else:
        if not (PROJECT / ".git").is_dir():
            raise RuntimeError("Project path exists but is not an Axral_vocal_AI Git checkout; move it before running.")
        status = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        if status.returncode:
            raise RuntimeError("Could not inspect existing Axral_vocal_AI checkout.")
        if status.stdout.strip():
            raise RuntimeError("Existing Axral_vocal_AI checkout has local changes; preserve them and use a clean Kaggle working directory.")
        origin_result = subprocess.run(
            ["git", "remote", "get-url", "--all", "origin"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        if origin_result.returncode:
            raise RuntimeError("Could not verify the Axral_vocal_AI origin remote.")
        origin_urls = [line.strip() for line in origin_result.stdout.splitlines() if line.strip()]
        if not is_trusted_axral_origin(origin_urls):
            raise RuntimeError("Existing checkout origin is not the trusted Axral_vocal_AI GitHub repository; refusing to fetch or reset it.")
        log("Verified existing checkout origin: nebula-INE/Axral_vocal_AI.")
        log("Refreshing existing Axral_vocal_AI checkout from origin/main.")
        fetch = subprocess.run(
            ["git", "fetch", "--depth", "1", "origin", "main"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        log(fetch.stdout)
        if fetch.returncode:
            raise RuntimeError("Could not refresh Axral_vocal_AI from origin/main.")
        reset = subprocess.run(
            ["git", "reset", "--hard", "FETCH_HEAD"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        log(reset.stdout)
        if reset.returncode:
            raise RuntimeError("Could not switch Axral_vocal_AI checkout to the fetched main revision.")

    # PR-only notebook users must be able to run before this pipeline is merged.
    # Prefer main after merge; use the feature branch only while required helpers are absent.
    missing_pipeline_files = [
        name for name in required_pipeline_files if not (PROJECT / name).is_file()
    ]
    if missing_pipeline_files:
        log(
            "Default branch does not contain the Kaggle pipeline helpers yet; "
            "trying feat/kaggle-one-url-pipeline."
        )
        fetch = subprocess.run(
            ["git", "fetch", "--depth", "1", "origin", "feat/kaggle-one-url-pipeline"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        log(fetch.stdout)
        if fetch.returncode:
            raise RuntimeError("Could not fetch the Kaggle pipeline feature branch.")
        reset = subprocess.run(
            ["git", "reset", "--hard", "FETCH_HEAD"],
            cwd=PROJECT, text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, errors="replace"
        )
        log(reset.stdout)
        if reset.returncode:
            raise RuntimeError("Could not switch to the Kaggle pipeline feature branch.")
        missing_pipeline_files = [
            name for name in required_pipeline_files if not (PROJECT / name).is_file()
        ]
    if missing_pipeline_files:
        raise RuntimeError(
            "Axral checkout is missing required Kaggle pipeline files: "
            + ", ".join(missing_pipeline_files)
        )
    source_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, errors="replace"
    )
    if source_commit.returncode:
        raise RuntimeError("Could not record Axral_vocal_AI source revision.")
    source_revision = source_commit.stdout.strip()
    log(f"Axral_vocal_AI source commit: {source_revision}")
    # The notebook kernel may start outside the repository directory.
    if str(PROJECT) not in sys.path:
        sys.path.insert(0, str(PROJECT))
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,training]"], output_tail=800)
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], output_tail=800)

    # Resolve HF_TOKEN from the environment or Kaggle Secrets. Never log the token.
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None  # Public datasets do not need a token.
    # The Hub token is passed only to the downloader subprocess. It is not
    # added to this kernel's environment or inherited by tests/training.
    download_env = os.environ.copy()
    download_env.pop("HF_TOKEN", None)
    if token:
        download_env["HF_TOKEN"] = token

    # Download only voicebank archives/metadata/WAVs and JSONL manifests.
    # The helper pins the Hub revision, enforces size limits, and safely extracts ZIP/tar archives.
    log("[1/5] Resolve and download Hugging Face dataset")
    run([
        sys.executable, "scripts/kaggle_hf_download.py", HF_DATA_URL,
        "--output-dir", str(DATA_ROOT),
    ], output_tail=0, redact_output=True, env=download_env)
    download_summary = json.loads(
        (DATA_ROOT / "download_summary.json").read_text(encoding="utf-8")
    )
    from scripts.voicebank_audit_summary import build_public_download_summary
    public_download_summary = build_public_download_summary(download_summary)
    (OUT / "download_summary.json").write_text(
        json.dumps(public_download_summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    log(
        "Dataset card license declaration: "
        + str(public_download_summary.get("dataset_card_license") or "not declared")
        + f"; license/readme documents indexed: {len(download_summary.get('license_documents', []))}"
    )
    dataset_id = public_download_summary.get("dataset")

    # Encoding safety: audit text is decoded by project reader (BOM/UTF-16/UTF-8/CP932).
    # Keep original voicebank files untouched; no destructive re-encoding.
    log("[2/5] Locate voicebank roots")
    from scripts.kaggle_hf_download import discover_voicebank_roots
    banks = discover_voicebank_roots(DATA_ROOT / "downloads", DATA_ROOT / "extracted")
    log(f"oto.ini roots: {len(banks)}")
    if not banks: raise RuntimeError("No oto.ini found. Check HF dataset layout/download.")

    log("[3/5] Voicebank audit (JSON per bank)")
    # Detailed reports contain aliases, sample filenames and local paths; keep them temporary.
    audit_dir = DATA_ROOT / "audit_reports"; audit_dir.mkdir(exist_ok=True)
    audit_results = []
    bank_fingerprints = {}
    for i, bank in enumerate(banks, 1):
        bank_relative_path = bank.relative_to(DATA_ROOT).as_posix()
        bank_fingerprints[i] = hashlib.sha256(
            bank_relative_path.casefold().encode("utf-8")
        ).hexdigest()[:16]
        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", bank_relative_path)[:100] or f"bank_{i}"
        target = audit_dir / f"{i:03d}_{safe}.json"
        audit_rc = run([sys.executable, "-m", "axral_vocal_ai.voicebank_audit_cli",
                        str(bank), "--output", str(target)], check=False, output_tail=0,
                       redact_output=True)
        audit_results.append((i, target, audit_rc))
        log(f"audit {i}/{len(banks)}: bank_index={i}, exit={audit_rc}")

    from scripts.voicebank_audit_summary import build_audit_index
    audit_index = build_audit_index(
        audit_results, bank_fingerprints=bank_fingerprints
    )
    audit_failures = audit_index["failures"]
    (OUT / "voicebank_audit_summary.json").write_text(
        json.dumps(audit_index, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    log(
        "Audit totals: "
        + json.dumps(audit_index["aggregate"], ensure_ascii=False)
        + f"; failed banks: {len(audit_failures)}"
    )

    log("[4/5] Tests")
    test_rc = run([sys.executable, "-m", "pytest", "-q"], check=False, output_tail=2500)
    log(f"pytest exit={test_rc}")

    from axral_vocal_ai.training_data import (
        MAX_JSONL_LINE_BYTES, MAX_MANIFEST_BYTES, MAX_MANIFEST_RECORDS,
        DatasetValidationError, validate_jsonl, validation_error_code,
    )
    from axral_vocal_ai.pitch_model import MAX_TRAINING_EXAMPLES, MAX_TRAINING_LABEL_POINTS
    training_limits = {
        "manifest_bytes": MAX_MANIFEST_BYTES,
        "jsonl_line_bytes": MAX_JSONL_LINE_BYTES,
        "manifest_records": MAX_MANIFEST_RECORDS,
        "label_points": MAX_TRAINING_LABEL_POINTS,
        "training_examples": MAX_TRAINING_EXAMPLES,
    }
    log("Training resource limits: " + json.dumps(training_limits, ensure_ascii=False))
    log("[5/5] Discover approved labelled training manifests")
    manifests = sorted(
        p for p in DATA_ROOT.rglob("*")
        if p.is_file() and p.suffix.casefold() == ".jsonl"
    )
    log(f"JSONL candidates: {len(manifests)}")
    eligible = []
    validation_failures = []
    validation_failure_codes = {}
    training_status = "blocked_no_approved_labelled_manifest"
    training_diagnostic_code = "manifest_not_found" if not manifests else "no_eligible_labelled_manifest"
    private_model_report = DATA_ROOT / "model_reports" / "pitch_delta_report.json"
    public_model_report = OUT / "models" / "pitch_delta_metrics.json"

    if download_summary.get("archive_errors"):
        training_status = "blocked_archive_extraction_errors"
        training_diagnostic_code = "archive_extraction_failed"
        log("Training blocked because one or more archives could not be safely extracted.")
    elif audit_failures:
        training_status = "blocked_voicebank_audit_findings"
        training_diagnostic_code = "voicebank_audit_findings"
        log(f"Training blocked because {len(audit_failures)} voicebank audit(s) returned findings.")
    elif test_rc != 0:
        training_status = "blocked_tests_failed"
        training_diagnostic_code = "test_suite_failed"
        log("Training blocked because pytest failed.")
    else:
        # Validate candidates in-process, but emit only stable diagnostic codes.
        # Exception details may contain voicebank names, record IDs, or local paths.
        for manifest_index, manifest in enumerate(manifests, 1):
            try:
                validate_jsonl(manifest, require_labeled=True, execution_context="hosted")
                eligible.append(manifest)
            except DatasetValidationError as exc:
                code = validation_error_code(exc)
                validation_failures.append({
                    "manifest_index": manifest_index,
                    "error_code": code,
                })
                validation_failure_codes[code] = validation_failure_codes.get(code, 0) + 1
                log(f"manifest {manifest_index}: validation failed ({code})")
            except (OSError, UnicodeError, ValueError):
                code = "manifest_validation_failed"
                validation_failures.append({
                    "manifest_index": manifest_index,
                    "error_code": code,
                })
                validation_failure_codes[code] = validation_failure_codes.get(code, 0) + 1
                log(f"manifest {manifest_index}: validation failed ({code})")

        if len(eligible) == 1:
            log("Using the only eligible labelled manifest; dataset path omitted from logs.")
            private_model_report.parent.mkdir(parents=True, exist_ok=True)
            # This is the existing metadata-only ridge MVP, not a waveform trainer.
            # The private trainer report contains split voicebank IDs; keep it temporary.
            run([
                sys.executable, "training/train_pitch_model.py",
                str(eligible[0]), "--output", str(private_model_report), "--execution-context", "hosted", "--evaluate-test"
            ], output_tail=2500)
            from scripts.voicebank_audit_summary import build_public_model_report
            private_report = json.loads(
                private_model_report.read_text(encoding="utf-8")
            )
            public_model_report.parent.mkdir(parents=True, exist_ok=True)
            public_model_report.write_text(
                json.dumps(
                    build_public_model_report(private_report),
                    ensure_ascii=False, indent=2
                ) + "\n",
                encoding="utf-8",
            )
            training_status = "metadata_pitch_model_trained_and_tested"
            training_diagnostic_code = "training_complete"
        elif len(eligible) > 1:
            training_status = "blocked_multiple_eligible_manifests"
            training_diagnostic_code = "multiple_eligible_manifests"
            log("Training blocked: multiple eligible manifests found; refusing to choose one implicitly.")
        elif not manifests:
            training_diagnostic_code = "manifest_not_found"
            log("No JSONL manifest was found in the selected Hugging Face dataset files.")
            log("UTAU oto.ini/WAV inventory does not contain aligned score-to-pitch-delta labels.")
            log("No model training was attempted; add a reviewed pitch_training_manifest.jsonl to the HF dataset.")
        else:
            training_diagnostic_code = "no_eligible_labelled_manifest"
            log("No eligible labelled JSONL manifest found.")
            log(
                "Manifest validation failure codes: "
                + json.dumps(validation_failure_codes, ensure_ascii=False, sort_keys=True)
            )
            log("No model training was attempted; review the manifest schema, permissions, split isolation, and pitch labels.")


    summary = {
        "dataset": dataset_id,
        "source_commit": source_revision,
        "hub_revision": download_summary.get("resolved_revision"),
        "download": public_download_summary,
        "voicebank_roots": len(banks),
        "audit_files": len(list(audit_dir.glob("*.json"))),
        "audit_failures": audit_failures,
        "voicebank_audit_summary": str(OUT / "voicebank_audit_summary.json"),
        "pytest_exit": test_rc,
        "jsonl_candidates": len(manifests),
        "eligible_manifests": len(eligible),
        "manifest_validation_failures": validation_failures,
        "manifest_validation_failure_codes": validation_failure_codes,
        "training": training_status,
        "training_diagnostic_code": training_diagnostic_code,
        "model_report": str(public_model_report) if public_model_report.exists() else None,
        "training_scope": "metadata_only_pitch_delta_ridge_mvp",
        "waveform_model_training": "not_implemented",
        "gpu_training_used": False,
        "training_limits": training_limits
    }
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    log(json.dumps(summary, ensure_ascii=False))
except Exception as exc:
    # Do not copy exception messages into artifacts: they may contain dataset
    # paths, filenames, or record identifiers. Keep only a stable type and
    # basename/line/function frames for debugging.
    error_name = re.sub(r"[^a-z0-9_]", "", type(exc).__name__.lower())[:60] or "unknown"
    log(f"pipeline failed: unexpected_{error_name}")
    safe_frames = [
        f"{Path(frame.filename).name}:{frame.lineno}:{frame.name}"
        for frame in traceback.extract_tb(exc.__traceback__)[-8:]
    ]
    if safe_frames:
        log("failure frames: " + " > ".join(safe_frames))
    raise
finally:
    log(f"LOG: {LOG}")
    log(f"OUTPUT: {OUT}")
    archive = OUT.parent / "axral_run_artifacts.zip"
    log(f"ARTIFACT_ZIP: {archive}")
    # Remove raw source data before creating any downloadable output.
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT, ignore_errors=True)
    log(f"TEMP_DATA_CLEANED: {not DATA_ROOT.exists()}")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        # Allowlist path-free aggregate reports, sanitized logs and model metrics.
        artifact_files = [
            LOG,
            OUT / "summary.json",
            OUT / "download_summary.json",
            OUT / "voicebank_audit_summary.json",
            OUT / "models" / "pitch_delta_metrics.json",
        ]
        for artifact in artifact_files:
            if artifact.is_file():
                bundle.write(artifact, artifact.relative_to(OUT.parent))
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(archive)))
    except Exception as display_error:
        log(f"Artifact link display unavailable ({type(display_error).__name__}).")

